# Earnings Press Report Predictive Transformer — end-to-end walkthroughAn encoder-only transformer adapted from *Attention Is All You Need* (Vaswani et al., 2017),applied to corporate earnings rather than translation.**Input:** the four GAAP figures in every earnings release — total revenue, net income,basic EPS, diluted EPS.**Output:** bullish / bearish / neutral for the next regular session's open→close move.No decoder. Sequence length 4. 42,535,683 trainable parameters.

In [ ]:
import json, os, sysimport numpy as npimport torchROOT = os.getcwd()sys.path.insert(0, os.path.join(ROOT, "model"))sys.path.insert(0, os.path.join(ROOT, "data"))from encoder import EarningsEncoder, CLASSES, param_countimport embed

## 1. The architectureEvery hyperparameter is read off the design spec rather than chosen:| spec | code ||---|---|| 12 heads, Q/K/V are 4×64 | `nhead=12`, d_model 768 → 768/12 = 64 || scale by √d_k | built into `nn.MultiheadAttention` || concat 12 heads × W^O (768×768) | the layer's internal `out_proj` || "add **then** normalize" | `norm_first=False` (post-LN) || ε = 1e-6 | `layer_norm_eps=1e-6` || 768 → 3072 → 768, ReLU | `dim_feedforward=3072` || repeats 6 times | 6 independently-initialised layers |

In [ ]:
model = EarningsEncoder()print(f"trainable parameters: {param_count(model):,}")model

### Self-test against the spec13 assertions covering every stage: input shape, the 4×4 attention matrix, W^O, bothfeed-forward matrices, layer independence, ε, post-LN ordering, head count, and thenumeric channel.

In [ ]:
from encoder import self_testself_test()

## 2. The input problemA sentence embedder is used to turn each metric into a 1×768 vector. But a sentenceembedder cannot represent **magnitude** — the templated sentences differ in only onetoken, and mean-pooling averages that difference into noise.The probe below embeds a ladder of revenue values spanning five orders of magnitude andmeasures cosine distance from the smallest rung. Near-zero spread means the model isblind to the number.

In [ ]:
embed.monotonicity_probe()

Max cosine distance across \$1M → \$100B is about **0.0031**, and not monotonic.Every company would collapse to nearly the same input matrix.The encoder is not at fault — it *is* sensitive to words (different metric wordings sit~0.44 apart) and it *is* sensitive to bare numbers (comma-grouped numerals span 0.21,monotonically). The template dilutes the number.### The fix: a separate numeric channelKeep the descriptive template so the model knows *which* metric a row is, and givemagnitude its own path — a per-metric learned projection of the scalar,$$\text{numeric}_j = s_j \cdot W_j + b_j, \qquad W_j, b_j \in \mathbb{R}^{768}$$summed into the text embedding, exactly as the 2017 paper sums positional encodings intotoken embeddings. The 4×768 shape entering the encoder is unchanged.

In [ ]:
recs = json.load(open("out/dataset.json"))recs.sort(key=lambda r: (r["react_date"], r["ticker"]))print(f"{len(recs)} labelled earnings prints\n")for r in recs[:3]:    print(f"  {r['ticker']:6s} {r['timing']:4s} react {r['react_date']}  "          f"rev {r['revenue']/1e6:>10,.0f}M  ni {r['net_income']/1e6:>9,.0f}M  "          f"eps {r['eps_basic']:>6.2f}/{r['eps_diluted']:<6.2f}  ret {r['ret']*100:+.2f}%")print("\ntemplates fed to the embedder:")for s in embed.templates(recs[0]):    print("  " + s)

In [ ]:
text = embed.embed_records(recs, verbose=False)raw = embed.scalars(recs)n_tr = int(len(recs) * 0.70)scaler = embed.fit_scaler(raw[:n_tr])          # train split only -- no leakagescal = embed.apply_scaler(raw, scaler)def spread(mat):    f = mat.reshape(len(mat), -1)    f = f / (np.linalg.norm(f, axis=1, keepdims=True) + 1e-9)    D = 1 - f @ f.T    return D[np.triu_indices(len(f), 1)].mean()with torch.no_grad():    combined = model.build_input(torch.from_numpy(text), torch.from_numpy(scal)).numpy()print(f"mean pairwise cos-distance between input matrices")print(f"  text only       {spread(text):.4f}")print(f"  text + numeric  {spread(combined):.4f}   ({spread(combined)/spread(text):.0f}x)")

## 3. Why the input is scaled by √d_modelAttention scores are $QK^\top/\sqrt{d_k}$. If the input rows are small, the scores arenear zero and softmax returns a uniform distribution — attention degenerates into plainaveraging.Layers 1–5 never have this problem because they receive **LayerNorm output**, whose rownorm is $\sqrt{768} = 27.71$ by construction. Layer 0 alone sees the raw embedding.Vaswani §3.4 multiplies embeddings by $\sqrt{d_{\text{model}}}$ for exactly this reason.

In [ ]:
T, S = torch.from_numpy(text[:32]), torch.from_numpy(scal[:32])print(f"{'layer':>6s} {'row norm in':>12s} {'attn deviation from uniform':>29s}")with torch.no_grad():    h = model.build_input(T, S)    for i, L in enumerate(model.layers):        print(f"{i:>6d} {float(h.norm(dim=2).mean()):>12.2f} "              f"{model.attention_uniformity(T, S, i):>29.4f}")        h = L(h)print(f"\nsqrt(d_model) = {768**0.5:.2f}")

## 4. TrainingForward pass → cross-entropy loss against the realised move → `loss.backward()` appliesthe chain rule back to every weight → Adam updates using its two moving averages(momentum $m_t$ and uncentered variance $v_t$), with bias correction.Adam β = (0.9, 0.98), ε = 1e-9, peak lr 3e-5, warmup then inverse-sqrt decay, 25 epochs.The embedder is frozen. Splits are time-ordered, never random.```bashpython train.py```

In [ ]:
ck = torch.load("model_deploy.pt", map_location="cpu", weights_only=False)model.load_state_dict({k: v.float() for k, v in ck["state"].items()})model.eval()lo, hi = ck["terciles"]print(f"classes: bearish <= {lo*100:+.2f}%  <  neutral  <  {hi*100:+.2f}% <= bullish")print("(raw open->close return of the first regular session after the release)")

## 5. Inference

In [ ]:
rec = recs[0]t = embed.embed_records([rec], verbose=False)s = embed.apply_scaler(embed.scalars([rec]), ck["scaler"])Tt, Ss = torch.from_numpy(t), torch.from_numpy(s)with torch.no_grad():    probs = torch.softmax(model(Tt, Ss), dim=1).numpy()[0]attn = model.first_layer_attention(Tt, Ss).numpy()[0]print(f"{rec['ticker']}")for f in embed.FIELDS:    v = rec[f]    print(f"  {f:12s} {v/1e6:>14,.1f} M" if "eps" not in f else f"  {f:12s} {v:>14,.2f}")print(f"\n  prediction: {CLASSES[int(probs.argmax())].upper()}")for c, p in zip(CLASSES, probs):    print(f"    {c:8s} {p*100:5.1f}%")

In [ ]:
import matplotlib.pyplot as pltshort = ["revenue", "net income", "basic EPS", "diluted EPS"]fig, ax = plt.subplots(figsize=(5, 4))im = ax.imshow(attn, cmap="Blues")ax.set_xticks(range(4), short, rotation=35, ha="right")ax.set_yticks(range(4), short)ax.set_xlabel("attends to"); ax.set_ylabel("from")ax.set_title("Layer-0 attention, mean over 12 heads")for i in range(4):    for j in range(4):        ax.text(j, i, f"{attn[i,j]:.3f}", ha="center", va="center", fontsize=8)fig.colorbar(im, fraction=0.045); fig.tight_layout(); plt.show()

Uniform 0.250 everywhere would mean attention is doing nothing. The spread here is learned.## 6. ScopeThe inputs are **absolute** GAAP levels with no expectation anchor — no consensus estimate,no prior quarter — so the model cannot distinguish a beat from a miss, and it is trained ona single earnings season. It demonstrates the architecture and the pipeline end to end.